# Automated Red Teaming Evaluation

This notebook is designed for app developers to automatically test the LLM application against security vulnerabilities (jailbreaks, prompt injection, hallucination, etc.) **without relying on any paid cloud services or tokens**. Everything is generated locally using your configured Google Gemini API key.

In [1]:
import os
import json
import pandas as pd
from IPython.display import display, Markdown
from dotenv import load_dotenv

# Load API keys and configurations
load_dotenv()

# Ensure promptfoo uses completely local generation (no promptfoo cloud)
os.environ["PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION"] = "1"
os.environ["PROMPTFOO_DISABLE_TELEMETRY"] = "1"

## 1. Run the Red Team Attacks
This will generate adversarial prompts using the configured local provider and evaluate your RAG application against them. The results are saved to `promptfoo_results.json`.

> Note: We use Python's `subprocess` to force the output to stream line-by-line in the notebook, bypassing Jupyter's output buffering.

In [ ]:
import subprocess
import sys

command = "npx promptfoo redteam run --env-file .env -c promptfooconfig.yaml -o promptfoo_results.json --max-concurrency 1 --delay 4000 --no-progress-bar --verbose"

# Start the process and stream the output line by line
process = subprocess.Popen(
    command, 
    shell=True, 
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT, 
    text=True, 
    bufsize=1
)

# Read output line by line as it is generated
for line in iter(process.stdout.readline, ''):
    print(line, end='')
    sys.stdout.flush()

process.stdout.close()
process.wait()

if process.returncode == 0:
    print("\n✅ Red team evaluation completed successfully!")
else:
    print(f"\n❌ Evaluation finished with exit code: {process.returncode}")

Loading environment variables from .env
[logger-DNHkR_3S.js:825] Verbose mode enabled via --verbose flag
[logger-DNHkR_3S.js:825] Generating test cases...
[logger-DNHkR_3S.js:825] Cache is disabled
[logger-DNHkR_3S.js:825] Setting default prompt because there is no `prompts` field
[logger-DNHkR_3S.js:825] Reading prompts from ["{{prompt}}"]
[logger-DNHkR_3S.js:825] plugins: hallucination, overreliance, rbac
[logger-DNHkR_3S.js:825] strategies: base64, leetspeak
[logger-DNHkR_3S.js:825] Extracted 1 target IDs from config providers: ["python:target_provider.py"]
[logger-DNHkR_3S.js:831] [RedteamProviderManager] Loading redteam provider
{
  "providedConfig": "google:gemini-2.5-flash-lite",
  "jsonOnly": false,
  "preferSmallModel": false
}
[logger-DNHkR_3S.js:831] Loading redteam provider
{
  "provider": "google:gemini-2.5-flash-lite"
}
[logger-DNHkR_3S.js:825] [RedteamProviderManager] Loaded redteam provider: google:gemini-2.5-flash-lite
[logger-DNHkR_3S.js:825] Synthesizing test cases f

## 2. Executive Summary
Parse the JSON results and display a high-level summary of the security posture right here in the notebook.

In [ ]:
def generate_summary(json_path):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        display(Markdown("**No results file found. Please run the evaluation cell above.**"))
        return

    results = data.get('results', {})
    evals = results.get('results', [])
    
    total_tests = len(evals)
    passed_tests = sum(1 for r in evals if r.get('success', False))
    failed_tests = total_tests - passed_tests
    
    display(Markdown("### Security Evaluation Results"))
    display(Markdown(f"- **Total Tests Run**: {total_tests}"))
    display(Markdown(f"- **Passed (Secure)**: {passed_tests}"))
    display(Markdown(f"- **Failed (Vulnerable)**: {failed_tests}"))
    
    summary_data = []
    for r in evals:
        # Extract plugin type and strategy from metadata
        plugin = r.get('metadata', {}).get('pluginId', 'Unknown')
        strategy = r.get('metadata', {}).get('strategyId', 'None')
        passed = r.get('success', False)
        summary_data.append({
            'Plugin (Vulnerability)': plugin,
            'Strategy': strategy,
            'Status': '✅ Passed' if passed else '❌ Failed (Vulnerable)'
        })
        
    if summary_data:
        df = pd.DataFrame(summary_data)
        # Group by Plugin to see failure rates
        grouped = df.groupby(['Plugin (Vulnerability)', 'Status']).size().unstack(fill_value=0)
        display(Markdown("### Vulnerability Breakdown"))
        display(grouped)
    else:
        display(Markdown("No detailed test results available."))

generate_summary('promptfoo_results.json')